# 从 Attention 到 TTT：线性化之路（1/5）—— Softmax Attention 到底在算什么

本系列 5 册基于苏剑林《从 Attention 到 TTT，重新理解 Attention 的线性化之路》
（存档见 `docs/attention-linearization-to-ttt.md`），面向**小白**：不要求你写过 Transformer，
每册配一段可以直接跑的 numpy 小例子，先建立直觉，再看公式。

| 册 | 主题 | 你会得到 |
|---|---|---|
| 1 | Softmax Attention 基础 | Q/K/V、打分加权求和、平方复杂度与 KV Cache |
| 2 | 分布视角 | 把 V 换成条件分布的期望，先消掉 V Cache |
| 3 | 流式更新 | 参数 W 作为 RNN 状态，边读边训（TTT 雏形） |
| 4 | 两次平均 | 两次近似把读出变成 o = μ_W(q)，看清每步放弃了什么 |
| 5 | 线性变体 | 线性化后得到 DeltaNet、MesaNet，状态大小对比 |

**这一册只做一件事**：把标准 Softmax Attention 的计算过程拆开，看清它为什么是平方复杂度、
为什么解码阶段被 KV Cache 卡住。

## 1. Q、K、V 分别是什么

把注意力想象成**查字典**：

- **Query（查询 $q$）**：当前这个 token 在"找什么"；
- **Key（键 $k_i$）**：第 $i$ 个历史 token 的"索引标签"；
- **Value（值 $v_i$）**：第 $i$ 个历史 token 真正携带的内容。

先拿 $q$ 和每个 $k_i$ 比对出一个相关分数，再用分数对所有 $v_i$ 加权平均，就得到当前位置的输出。

$$a_i = \frac{e^{q \cdot k_i}}{Z}, \qquad Z = \sum_{j=1}^{n} e^{q \cdot k_j}, \qquad o = \sum_{i=1}^{n} a_i v_i$$

$a_i$ 就是第 $i$ 个历史 token 被"注意"到的权重，softmax 保证所有权重加起来等于 1。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n, d = 3, 4  # 3 个 token，每个向量 4 维（教学规模，真实模型是几十到上百维）

# 模拟 3 个历史 token 的键和值，以及当前 token 的查询
K = rng.standard_normal((n, d))
V = rng.standard_normal((n, d))
q = rng.standard_normal(d)

# 1) 打分：q 和每个 k_i 做点积
scores = K @ q
print("原始分数:      ", scores.round(3))

# 2) softmax：分数 -> 加起来等于 1 的权重
def softmax(x):
    x = x - x.max()          # 数值稳定
    e = np.exp(x)
    return e / e.sum()

weights = softmax(scores)
print("softmax 权重:  ", weights.round(3), " 合计 =", weights.sum().round(6))

# 3) 加权求和：按权重把所有 v_i 混合成输出
o = weights @ V
print("输出 o:        ", o.round(3))

## 2. 复杂度：平方项从哪里来

对当前位置 $t$，要让 $q_t$ 跟前面**每一个** $k_1,\dots,k_{t-1}$ 配对打分。
长度为 $n$ 的序列，所有位置两两配对，打分次数是：

- **预填充（prefill）**：整段输入一次性并行算，注意力矩阵是 $n \times n$，计算量 $O(n^2 d)$；
- **解码（decode）**：每生成一个新 token，都要把全部历史 K、V 再读一遍，
  历史越长，每一步读写的数据越多。

下面用一段代码直观比较不同序列长度下，配对打分的次数增长。

In [ ]:
print(f"{'序列长度 n':>10} | {'配对次数 n*n':>12} | {'相对 n=8':>10}")
print("-" * 38)
for n in (8, 16, 32, 64, 128):
    print(f"{n:>10} | {n*n:>12} | {n*n/64:>10.1f}x")

## 3. KV Cache：解码阶段的显存瓶颈

解码时模型不能每次都把全部历史重算一遍，于是把每个历史 token 的 $k_i, v_i$ **缓存**下来，
这就是 **KV Cache**：

- 缓存体积随序列长度**线性增长**：$O(n \cdot d)$ 的 K 和同样一份 V；
- 解码每生成一个 token，都要把这份缓存读一遍，显存带宽和容量成为主要瓶颈；
- 上下文拉到几十万、上百万 token 时，这一项比计算量更先顶到天花板。

文章里那句话的意思是：**绝大多数线性化注意力，本质上都是在用不同思路压缩 KV Cache。**
后面几册走的"平均之路"，就是从压缩缓存这个目标出发的。

## 4. 小结（1/5）

- Attention = 用 $q$ 和每个 $k_i$ 打分 → softmax 得权重 $a_i$ → 对 $v_i$ 加权求和；
- 两两打分带来 $O(n^2 d)$ 的计算复杂度；
- 解码阶段真正难受的是随长度线性增长的 **KV Cache**；
- 线性化的动机：把历史压进固定大小的状态，而不是逐 token 缓存 K、V。

→ **下一册**：不直接背下 $v_i$，把它看成条件分布的期望，看看怎么先把 V Cache 消掉。